# Onset-HFO 2 — An evidence-only agent on an open-weight model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/bci-gan/blob/master/onset-hfo/notebooks/02_agentic_analysis.ipynb)

Notebook 1 produced tables. This notebook puts a language model in front of
them — carefully.

**The design in one paragraph.** The agent can call eight read-only tools
over one saved analysis. It cannot run a detector, change a threshold, open a
file or reach the network. Every factual claim it makes must cite an
`evidence_id` that a tool actually returned, and every number it states must
appear in a tool result. Those two checks run *after* the model answers; if
either fails, the answer is discarded and the agent says it cannot answer.
Questions about treatment, diagnosis or another patient are refused *before*
the model is called at all.

**Why bother with an agent then?** Because choosing which question to ask of a
dataset — "is this channel's lead real, or do the detectors disagree about
it?" — is exactly the kind of multi-step, ill-specified work a language model
is good at, and exactly the kind of work that must never be allowed to invent
a number. The split is: the model decides *what to look up and how to say
it*; the pipeline decides *what is true*.

Everything runs on **open weights** (Qwen2.5-Instruct by default). Nothing in
this project requires a hosted proprietary model.

In [1]:
# Colab setup. On your own machine, skip this cell and run
#   pip install -e ".[dev]"  from the onset-hfo directory instead.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/berdakh/bci-gan.git"
BRANCH = "master"   # the repository default branch

if IN_COLAB and not os.path.exists("bci-gan"):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1", REPO], check=True)
if IN_COLAB:
    os.chdir("/content/bci-gan/onset-hfo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

working directory: /home/user/bci-gan/onset-hfo


## 1. Get an analysis to talk about

If you ran notebook 1, its results are already on disk. If not, this cell
builds one from the synthetic recording in a few seconds — no download.

In [2]:
import os
from onset_hfo.store import ResultStore

REAL = "artifacts/results/sub-pt01_ictal_run-01"
if os.path.exists(os.path.join(REAL, "events.csv")):
    results_dir = REAL
else:
    from onset_hfo.pipeline import run_pipeline
    from onset_hfo.synthetic import make_synthetic_recording
    recording = make_synthetic_recording(duration_s=60)
    results_dir = str(run_pipeline(recording).save("artifacts/results"))

store = ResultStore(results_dir)
store

ResultStore(sub-pt01_ictal_run-01: sub-pt01, 4609 events, detectors=['line_length', 'rms', 'spike'])

## 2. What the agent can see

The store is the whole world the agent has access to. Look at the tools it
exposes — the descriptions below are literally what the model is shown.

In [3]:
from onset_agent.tools import TOOLS

for name, tool in TOOLS.items():
    args = ", ".join(tool.parameters["properties"]) or "-"
    print(f"{name:26s} args: {args}")
    print(f"{'':26s} {tool.description[:110]}...")

get_recording_metadata     args: -
                           What was analysed: subject, source dataset, sampling rate, analysed time window, seizure markers, detectors us...
list_channels              args: detector
                           Every analysed channel with its event count and rate for one detector, ordered by rate. Use when asked what wa...
top_channels               args: detector, k
                           The channels with the highest event rate, with 95% confidence intervals. Use for 'which channels stand out', '...
channel_summary            args: channel
                           Everything known about one channel: each detector's rate, rank, confidence interval, mean frequency and amplit...
get_evidence               args: channel, detector, k
                           The strongest detected events on a channel: time windows, peak frequency, spectral prominence and amplitude, e...
detector_disagreements     args: -
                           Channels the two

In [4]:
# There is no tool that takes a patient argument: scope belongs to the
# application, not to model output. And there is no tool that writes anything.
import json
print(json.dumps(TOOLS["get_evidence"].schema(), indent=2))

{
  "type": "function",
  "function": {
    "name": "get_evidence",
    "description": "The strongest detected events on a channel: time windows, peak frequency, spectral prominence and amplitude, each with an evidence_id. Every factual claim in an answer must cite one of these ids.",
    "parameters": {
      "type": "object",
      "properties": {
        "channel": {
          "type": "string"
        },
        "detector": {
          "type": "string",
          "enum": [
            "rms",
            "line_length",
            "spike"
          ]
        },
        "k": {
          "type": "integer",
          "minimum": 1,
          "maximum": 10
        }
      },
      "required": [
        "channel"
      ],
      "additionalProperties": false
    }
  }
}


## 3. First, with no model at all

The `scripted` backend is a deterministic keyword policy that speaks the same
tool-calling protocol. It is **not** a language model — it exists so the
machinery can be demonstrated and tested offline, and so you can see the loop
before any weights are downloaded.

In [5]:
from onset_agent import OnsetAgent, make_backend

agent = OnsetAgent(store, make_backend("scripted"))
answer = agent.ask("Which channels have the highest ripple rate?")
print(answer)

Highest rates (line_length): PST2-PST3 83.0/min (n=83); ATT7-ATT8 82.0/min (n=82); ATT6-ATT7 82.0/min (n=82). Rates are measurements, not a seizure-onset zone.
  tools: top_channels


In [6]:
answer = agent.ask("What is the evidence for the top channel?")
print(answer.text)
print("\ncitations:", answer.evidence_ids)
print("resolved:", [store.resolve(e) is not None for e in answer.evidence_ids])

Highest rates (line_length): PST2-PST3 83.0/min (n=83); ATT7-ATT8 82.0/min (n=82); ATT6-ATT7 82.0/min (n=82). Evidence on PST2-PST3: 104.23-104.336 s, peak 192.3828 Hz; 107.471-107.627 s, peak 187.9883 Hz. Rates are measurements, not a seizure-onset zone.

citations: ['sub-pt01|PST2-PST3|line_length|104.230', 'sub-pt01|PST2-PST3|line_length|107.471']
resolved: [True, True]


## 4. Now with a real open-weight model

Two ways, pick one.

**A. Ollama** (best on your own machine, also works in Colab):

```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama pull qwen2.5:7b-instruct
ollama serve &
```

then `make_backend("ollama", model="qwen2.5:7b-instruct")`.

**B. Hugging Face transformers** (in-process, no server — the easy path in
Colab). On a free CPU instance use the 1.5B model; with a GPU runtime use the
7B, which is much better at tool calling.

The cell below tries transformers and falls back to the scripted policy if the
packages or the weights are unavailable, so the notebook always runs.

In [7]:
USE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # try "Qwen/Qwen2.5-7B-Instruct" on a GPU runtime

backend = None
try:
    import torch, transformers  # noqa: F401
    backend = make_backend("transformers", model=USE_MODEL)
    print("using", backend.describe())
except Exception as exc:
    print("could not load a language model:", type(exc).__name__, exc)
    print("falling back to the scripted policy (install with: pip install -e '.[llm]')")
    backend = make_backend("scripted")

llm_agent = OnsetAgent(store, backend, verbose=True)

could not load a language model: ModuleNotFoundError No module named 'torch'
falling back to the scripted policy (install with: pip install -e '.[llm]')


In [8]:
answer = llm_agent.ask("Which channels have the highest ripple rate, and how sure can we be?")
print("\n", answer.text)
print("tools:", answer.tools_called, "| verified:", answer.verified)

  [tool] top_channels({'k': 5}) -> ok

 Highest rates (line_length): PST2-PST3 83.0/min (n=83); ATT7-ATT8 82.0/min (n=82); ATT6-ATT7 82.0/min (n=82). Rates are measurements, not a seizure-onset zone.
tools: ['top_channels'] | verified: True


If you are running the 1.5B model, there is a good chance the answer above was
*refused* rather than produced. That is not a bug in the notebook — it is the
system working. A small model fumbles the tool protocol or states a number it
did not retrieve, the guards catch it, and the agent declines instead of
publishing a plausible fabrication. Swap in the 7B model and the same
questions start going through.

This is the most useful thing this notebook can show you: **what the failure
looks like, and that it is contained.**

## 5. The refusals

Three kinds, all checked before the model runs.

In [9]:
for question in ["Which contacts should we resect?",
                 "Does this patient have epilepsy?",
                 "What did you find in patient sub-pt02?",
                 "Where do the seizures start?"]:
    a = agent.ask(question)
    print(f"Q: {question}\n   [{'refused' if a.refused else 'answered'}] {a.text}\n"
          f"   tools called: {a.tools_called or 'none'}\n")

Q: Which contacts should we resect?
   [refused] I do not answer questions about treatment or what should be done. I can show which channels carry the most detected activity and the signal windows behind that; the clinical decision is the team's.
   tools called: none

Q: Does this patient have epilepsy?
   [refused] I cannot diagnose, give a prognosis, or say where seizures start. A high event rate is a measurement, not a seizure-onset zone. I can tell you what the detectors measured and where they disagree.
   tools called: none

Q: What did you find in patient sub-pt02?
   [refused] I can only see the saved analysis of sub-pt01. Open that patient's results to ask about their channels.
   tools called: none

Q: Where do the seizures start?
   [refused] I cannot diagnose, give a prognosis, or say where seizures start. A high event rate is a measurement, not a seizure-onset zone. I can tell you what the detectors measured and where they disagree.
   tools called: none



Note the last line of each: no tool was called. The refusal costs nothing and
cannot be argued out of, because there is no model in the path to argue with.

## 6. Watching a guard catch a fabrication

Let us make a model that lies, and check that the system holds. This backend
retrieves real data and then states a rate nobody measured.

In [10]:
import json
from onset_agent.backends import AssistantMessage, Backend, ToolCall

class LyingBackend(Backend):
    name = "deliberately-wrong"
    def __init__(self): self.turn = 0
    def chat(self, messages, tools):
        self.turn += 1
        if self.turn == 1:
            return AssistantMessage(tool_calls=[ToolCall("top_channels", {"k": 3}, "c1")])
        return AssistantMessage(content=json.dumps({
            "answer": "The leading channel fires at 999.9 events/min, far above the others.",
            "evidence_ids": ["sub-xx|MADE-UP|rms|0.000"]}))

bad = OnsetAgent(store, LyingBackend())
answer = bad.ask("How often does the top channel fire?")
print(answer.text, "\n")
print(json.dumps([s for s in answer.trace if s["type"] == "answer"], indent=2))

I could not produce an answer I can stand behind: the checks on citations and numbers did not pass. The saved report and tables are authoritative; see report.md. 

[
  {
    "type": "answer",
    "step": 1,
    "verified": false,
    "problems": [
      "citation 'sub-xx|MADE-UP|rms|0.000' was never returned by a tool",
      "the value '999.9 /min' does not appear in any tool result"
    ]
  },
  {
    "type": "answer",
    "step": 2,
    "verified": false,
    "problems": [
      "citation 'sub-xx|MADE-UP|rms|0.000' was never returned by a tool",
      "the value '999.9 /min' does not appear in any tool result"
    ]
  },
  {
    "type": "answer",
    "step": 3,
    "verified": false,
    "problems": [
      "citation 'sub-xx|MADE-UP|rms|0.000' was never returned by a tool",
      "the value '999.9 /min' does not appear in any tool result"
    ]
  }
]


Both problems are named: the citation was never returned by a tool, and the
number appears in no tool result. The answer is dropped. Note what the agent
says instead — it points at the report, which *is* authoritative.

## 7. The trace

Every answer carries the full sequence of tool calls, arguments and
verification outcomes. This is what you would show a reviewer who asks "where
did this sentence come from?".

In [11]:
answer = agent.ask("Where do the two detectors disagree?")
print(answer.text, "\n")
print(json.dumps(answer.as_dict(), indent=2)[:1800])

The two detectors rank 7 leading channel(s) very differently: PST2-PST3, AD1-AD2, AD3-AD4. Rates are measurements, not a seizure-onset zone. 

{
  "question": "Where do the two detectors disagree?",
  "answer": "The two detectors rank 7 leading channel(s) very differently: PST2-PST3, AD1-AD2, AD3-AD4. Rates are measurements, not a seizure-onset zone.",
  "refused": false,
  "reason": "",
  "evidence_ids": [],
  "tools_called": [
    "detector_disagreements"
  ],
  "backend": "scripted (no language model)",
  "verified": true,
  "trace": [
    {
      "type": "tool_call",
      "tool": "detector_disagreements",
      "arguments": {},
      "ok": true
    },
    {
      "type": "answer",
      "step": 1,
      "verified": true,
      "problems": []
    }
  ]
}


## 8. The full demonstration set

In [12]:
from onset_agent.prompts import EXAMPLE_QUESTIONS

for a in agent.ask_many(EXAMPLE_QUESTIONS):
    status = "REFUSED" if a.refused else "answer "
    print(f"[{status}] {a.question}\n           {a.text[:220]}\n")

[answer ] Which channels have the highest ripple rate?
           Highest rates (line_length): PST2-PST3 83.0/min (n=83); ATT7-ATT8 82.0/min (n=82); ATT6-ATT7 82.0/min (n=82). Rates are measurements, not a seizure-onset zone.

[answer ] What is the evidence for the top channel?
           Highest rates (line_length): PST2-PST3 83.0/min (n=83); ATT7-ATT8 82.0/min (n=82); ATT6-ATT7 82.0/min (n=82). Evidence on PST2-PST3: 104.23-104.336 s, peak 192.3828 Hz; 107.471-107.627 s, peak 187.9883 Hz. Rates are meas

[answer ] Where do the two detectors disagree?
           The two detectors rank 7 leading channel(s) very differently: PST2-PST3, AD1-AD2, AD3-AD4. Rates are measurements, not a seizure-onset zone.

[answer ] Did the event rate change during the seizure?
           Rate before vs during the marked seizure: PST2-PST3 0.0/min -> 146.2555/min; ATT7-ATT8 0.0/min -> 144.4934/min. Rates are measurements, not a seizure-onset zone.

[answer ] What are the limitations of this analysis?
     

## What to try next

* Ask something the tools genuinely cannot answer ("what is the patient's
  age?") and watch it decline rather than guess.
* Add a tool in `onset_agent/tools.py` — say, one that returns the rejected
  events and their reasons — and see the model start using it. The schema is
  the only thing the model knows about it.
* Try a different open-weight model (`llama3.1:8b-instruct`,
  `mistral-nemo`) and compare how often the guards fire. That number is a
  useful, cheap benchmark of a model's tool discipline.
* Read `docs/AGENT.md` for the threat model — including why tool *results*
  are treated as data and never as instructions.